### __EXERCISE 4.1: Naive Bayes Classifier__  

#### Name: __Escrupulo, Ma. Asherah Francince Faith S.__
#### Name: __Porras, Jessie Loraine P.__

#### Course, Year and Section: __BSCS 3-B AI__
#### Date: __2/25/2026__

---
1. Performing it manually. In manually developing a Naïve Bayes model, create methods that will do the following:

    a. Generate a Bag of Words (for word frequency)

    b. Calculate the prior for the class HAM and SPAM

    c. Calculate the likelihood of the tokens in the vocabulary with respect to the class.

    d. Determine the class of the following test sentence:

        i. Limited offer, click here!

        ii. Meeting at 2 PM with the manager.


In [2]:
import math
import random
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
### Creating the document with labels (SPAM or HAM)

import string


document = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you sent the report?", "HAM")
]

# Pre-processing for the Bag of Words of the document

    # Getting the list of stop words in English
stop_words = set(stopwords.words('english'))


# Preprocessing function to clean and tokenize the document
def preprocess_document(text):

    # Converting to lowercase
    text = text.lower()

    # Removing punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenizing
    words = word_tokenize(text)

    #remove stopwords
    filtered_words = [word for word in words if word.lower() not in stop_words]

    return filtered_words

# Applying preprocessing to each document and printing the results

for text, label in document:
    processed_document = preprocess_document(text)
    print(f"Original Text: {text}")
    print(f"Processed Words: {processed_document}")
    print("---------------------------------")

processed_document = []
for text, label in document:
    processed_words = preprocess_document(text)
    processed_document.append((processed_words, label))

print("Processed Document with Labels:")
for words, label in processed_document:
    print(f"Words: {words}, Label: {label}")


Original Text: Free money now!!!
Processed Words: ['free', 'money']
---------------------------------
Original Text: Hi mom, how are you?
Processed Words: ['hi', 'mom']
---------------------------------
Original Text: Lowest price for your meds
Processed Words: ['lowest', 'price', 'meds']
---------------------------------
Original Text: Are we still on for dinner?
Processed Words: ['still', 'dinner']
---------------------------------
Original Text: Win a free iPhone today
Processed Words: ['win', 'free', 'iphone', 'today']
---------------------------------
Original Text: Let's catch up tomorrow at the office
Processed Words: ['lets', 'catch', 'tomorrow', 'office']
---------------------------------
Original Text: Meeting at 3PM tomorrow
Processed Words: ['meeting', '3pm', 'tomorrow']
---------------------------------
Original Text: Get 50% off, limited time!
Processed Words: ['get', '50', 'limited', 'time']
---------------------------------
Original Text: Team meeting in the office
Proc

In [4]:
# Creating Vocabulary

import re

# importing defaultdict from collections module to handle word frequency counting
from collections import (
    defaultdict,
)

print("Processed documents preview:", processed_document[:3])

# Initializing a defaultdict with integer values to store word frequencies
vocab = defaultdict(int)

# Looping through each document in the processed dataset
for words, label in processed_document:
    
    # Updating the vocabulary with word frequencies
    for word in words:
        vocab[word] += 1

# List of all unique words in the vocabulary
vocab_list = list(vocab.keys())
print("Number of unique words in the vocabulary:", len(vocab_list))
print(vocab_list)

Processed documents preview: [(['free', 'money'], 'SPAM'), (['hi', 'mom'], 'HAM'), (['lowest', 'price', 'meds'], 'SPAM')]
Number of unique words in the vocabulary: 27
['free', 'money', 'hi', 'mom', 'lowest', 'price', 'meds', 'still', 'dinner', 'win', 'iphone', 'today', 'lets', 'catch', 'tomorrow', 'office', 'meeting', '3pm', 'get', '50', 'limited', 'time', 'team', 'click', 'prizes', 'sent', 'report']


In [5]:
# a. Generate a Bag of Words (for word frequency)

# Calculating word frequencies and then vectorize

vocab_index = {word: idx for idx, word in enumerate(vocab_list)}

def create_bow_vector(doc_tokens, vocab_list):
    vector = [0] * len(vocab_list)  # Initialize a vector of zeros
    
    for word in doc_tokens:
        if word in vocab_index:
            vector[vocab_index[word]] += 1              
    return vector

# Creating BoW vectors for each document in the processed dataset
bow_vectors = [create_bow_vector(words, vocab_list) for words, label in processed_document]
print("Bag of Words Vectors:")
for vector in bow_vectors:
    print(vector)

Bag of Words Vectors:
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]


In [6]:
# b.	Calculate the prior for the class HAM and SPAM

from collections import Counter

# Extracting the labels
labels = [label for __, label in processed_document]

# Count how many documents per class
label_count = Counter(labels)

# Total Number of documents
total_docs = len(processed_document)

# Calculating priors
priors = {label: count / total_docs for label, count in label_count.items()}

print("Class Priors:")
for label, prob in priors.items():
    print(f"{label}: {prob:.2f}")

Class Priors:
SPAM: 0.45
HAM: 0.55


In [7]:
# c.	Calculate the likelihood of the tokens in the vocabulary with respect to the class.

vocab_size = len(vocab_list)
alpha = 1   # applying Laplace smoothing

# initializing counters
word_cnt_per_class = {label: np.zeros(vocab_size) for label in label_count}
total_words_per_class = {label: 0 for label in label_count}

# Counting word occurences per class

for tokens, label in processed_document:
    for word in tokens:
        if word in vocab_index:
            idx = vocab_index[word]
            word_cnt_per_class[label][idx] += 1
            total_words_per_class[label] += 1

likelihoods = {}
for label in label_count:
    likelihoods[label] = (word_cnt_per_class[label] + alpha) / (total_words_per_class[label] + alpha * vocab_size)


# Example implementation with the word "free"
print(f"P('{vocab_list[0]}'|SPAM) = {likelihoods['SPAM'][0]:.4f}")
print(f"P('{vocab_list[0]}'|HAM) = {likelihoods['HAM'][0]:.4f}")



P('free'|SPAM) = 0.0714
P('free'|HAM) = 0.0233


In [8]:
# Creating Naive Bayes function

def naive_bayes(test_tokens, priors, likelihoods, vocab_index):
    scores = {}

    for label in priors: 
        score = np.log(priors[label])
        for word in test_tokens:
            if word in vocab_index:
                idx = vocab_index[word]
                score += np.log(likelihoods[label][idx])
        scores[label] = score

    predicted_class = max(scores, key=scores.get)
    return predicted_class, scores

In [9]:
# d.	Determine the class of the following test sentence:
        # i.	Limited offer, click here!
        # ii.	Meeting at 2 PM with the manager.

test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

for sentence in test_sentences:
    tokens = preprocess_document(sentence)
    predicted_class, scores = naive_bayes(tokens, priors, likelihoods, vocab_index)

    print(f"Sentence: {sentence}")
    print(f"Tokens: {tokens}")
    print(f"Predicted Class: {predicted_class}")

    format_scores = {label: f"{score:.3f}" for label, score in scores.items()}
    print(f"Scores: {format_scores}")
    print("------------------------------")

Sentence: Limited offer, click here!
Tokens: ['limited', 'offer', 'click']
Predicted Class: SPAM
Scores: {'SPAM': '-6.878', 'HAM': '-8.129'}
------------------------------
Sentence: Meeting at 2 PM with the manager.
Tokens: ['meeting', '2', 'pm', 'manager']
Predicted Class: HAM
Scores: {'SPAM': '-4.526', 'HAM': '-3.269'}
------------------------------


---
2. Using Scikit-Learn. Use the scikit-learn package to train and test a Multinomial Naïve Bayes classifer.
  
    a. Determine the class of the following test sentence:
  
        i.  Limited offer, click here!
        ii. Meeting at 2 PM with the manager.

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

In [28]:
#Training and Testing 
train_sentences = [
    "Free money now!!!",
    "Hi mom, how are you?",
    "Lowest price for your meds", 
    "Are we still on for dinner?", 
    "Win a free iPhone today", 
    "Let's catch up tomorrow at the office",
    "Meeting at 3PM tomorrow", 
    "Get 50% off, limited time!",
    "Team meeting in the office", 
    "Click here for prizes!", 
    "Can you sent the report?"
]

train_labels = ["SPAM", "HAM", "SPAM", "HAM", "SPAM", "HAM", "HAM", "SPAM", "HAM", "SPAM", "HAM"]

In [29]:
# Converting text to numerical features
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_sentences)

In [30]:
# Train the Multinomial Naive Bayes model
model = MultinomialNB()
model.fit(X_train, train_labels)

MultinomialNB()

In [31]:
# Preparing the test sentences
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

X_test = vectorizer.transform(test_sentences)


In [32]:
# Predict what type of class the test sentences are 
predictions = model.predict(X_test)

In [35]:
# Output - Predicted class for each sentence
for sentence, label in zip(test_sentences, predictions):
    print(f"Sentence: {sentence}")
    print(f"Predicted Class: {label}")

Sentence: Limited offer, click here!
Predicted Class: SPAM
Sentence: Meeting at 2 PM with the manager.
Predicted Class: HAM
